In [2]:
import pandas as pd
import numpy as np
import time

# Read input data
data = pd.read_excel('/Users/workk/Downloads/input7u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Downloads/power_demand7u.xlsx')
power_demand = power_demand_data['load'].values

# Hybrid EMFO-SCA Parameters
num_particles = 50
num_iterations = 100
alpha = 0.5  # Attraction coefficient for EMFO
beta = 0.5   # Sine-Cosine factor for SCA
sigma = 0.5  # Decay factor for C component (EMFO)

# Define Hybrid EMFO-SCA-based power calculation
def calculate_power_hybrid(demand, a, b, c, pmin, pmax, rampup, rampdown):
    num_units = len(pmin)

    # Initialize particle positions and velocities
    particle_positions = np.random.uniform(pmin, pmax, (num_particles, num_units))
    global_best_position = np.zeros(num_units)
    global_best_score = float('inf')

    for iteration in range(num_iterations):
        for i in range(num_particles):
            # Ensure particles respect generation limits
            particle_positions[i] = np.clip(particle_positions[i], pmin, pmax)

            # Calculate fuel cost and penalty for demand mismatch
            squared_P = np.multiply(particle_positions[i], particle_positions[i])
            fuel_cost = np.sum(a + b * particle_positions[i] + c * squared_P)
            power_output = np.sum(particle_positions[i])
            mismatch_penalty = abs(demand - power_output) * 10000  # High penalty for mismatch
            score = fuel_cost + mismatch_penalty

            # Update global best
            if score < global_best_score:
                global_best_score = score
                global_best_position = particle_positions[i].copy()

        # Update particle positions using EMFO equations
        for i in range(num_particles):
            if i != np.argmin([global_best_score]):  # Avoid updating the global best particle
                r1, r2 = np.random.rand(num_units), np.random.rand(num_units)
                A = alpha * np.exp(-sigma * iteration / num_iterations) * (2 * r1 - 1)
                B = alpha * np.exp(-sigma * iteration / num_iterations) * (2 * r2 - 1)
                C = np.exp(-sigma * iteration / num_iterations) * np.cos(2 * np.pi * r1)
                particle_positions[i] += A * (global_best_position - particle_positions[i]) + B * C

                # Ensure the solution is within valid range
                particle_positions[i] = np.clip(particle_positions[i], pmin, pmax)

        # Update particle positions using SCA equations
        for i in range(num_particles):
            r1 = np.random.uniform()
            r2 = np.random.uniform()
            r3 = np.random.uniform()
            if r1 < 0.5:
                particle_positions[i] += beta * np.sin(2 * np.pi * r3) * (global_best_position - particle_positions[i]) * r2
            else:
                particle_positions[i] += beta * np.cos(2 * np.pi * r3) * (global_best_position - particle_positions[i]) * r2

        # Normalize the global best to meet demand
        total_power = np.sum(global_best_position)
        if total_power != 0:
            scaling_factor = demand / total_power
            global_best_position = global_best_position * scaling_factor

        # Enforce generation limits on the global best
        global_best_position = np.clip(global_best_position, pmin, pmax)

        # Termination condition for power balance
        if abs(demand - np.sum(global_best_position)) < 0.0001:
            break

    return global_best_position, iteration + 1

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = a * squaredP + b * P + c 
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values
    P, iterations = calculate_power_hybrid(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(P),
                        'Total Fuel Cost': total_Fuel_Cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Economic Dispatch/Output/EMFO-SCA (ED).xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'EMFO-SCA (ED).xlsx'")

Load Counter: 1
Output for Power Demand 181.54 MW
   Unit  Power Produced (MW)
0     1             4.392955
1     2            25.615187
2     3             6.035240
3     4            64.693975
4     5            18.546279
5     6            60.441103
6     7             1.819130
Total Power Produced: 181.54 MW
Total Fuel Cost: Rp 126404.89465595908
Computation Time: 0.0016179084777832031 seconds
------------------------------------------------

Load Counter: 2
Output for Power Demand 209.07 MW
   Unit  Power Produced (MW)
0     1             2.490025
1     2            25.058166
2     3             6.041138
3     4            72.753049
4     5            35.850422
5     6            65.306250
6     7             1.569927
Total Power Produced: 209.07 MW
Total Fuel Cost: Rp 159850.0795107302
Computation Time: 0.00150299072265625 seconds
------------------------------------------------

Load Counter: 3
Output for Power Demand 160.10 MW
   Unit  Power Produced (MW)
0     1             4.